# freqgen Colab v2 — Correct SPAI Setup (matches paper's original env)

**Key lesson from v1:** SPAI requires `numpy~=1.26.4`. Never upgrade numpy.
Install SPAI requirements FIRST, then do everything else in that environment.

**Inference command (from README):** `python -m spai infer --input <dir> --output <dir>`
No `--cfg` needed — defaults to `./weights/spai.pth` and `./configs/spai.yaml`.

Runtime: `Runtime → Change runtime type → T4 GPU`

## 1. Install SPAI (numpy 1.26.4, matches paper)

In [1]:
import subprocess, sys

# Step 1: clone SPAI
import os
if not os.path.exists('/content/spai'):
    !git clone https://github.com/mever-team/spai.git /content/spai -q
%cd /content/spai

# Step 2: install PyTorch first (conda-style, via pip with CUDA)
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q

# Step 3: install SPAI requirements EXACTLY as specified (numpy 1.26.4)
# Do NOT upgrade numpy — SPAI requires ~=1.26.4
!pip install -r requirements.txt filetype -q

import numpy as np
print(f'numpy {np.__version__}  (should be 1.26.x)')
import torch
print(f'torch {torch.__version__}  cuda={torch.cuda.is_available()}')
print('SPAI install OK')

/content/spai
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.5/869.5 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 9.6 MB/s eta

## 2. Download SPAI weights

In [2]:
import os
os.makedirs('/content/spai/weights', exist_ok=True)
if not os.path.exists('/content/spai/weights/spai.pth'):
    !gdown 1vvXmZqs6TVJdj8iF1oJ4L_fcgdQrp_YI -O /content/spai/weights/spai.pth
sz = os.path.getsize('/content/spai/weights/spai.pth') // 1_000_000
print(f'Weights: {sz} MB  (expect ~934)')

Downloading...
From (original): https://drive.google.com/uc?id=1vvXmZqs6TVJdj8iF1oJ4L_fcgdQrp_YI
From (redirected): https://drive.google.com/uc?id=1vvXmZqs6TVJdj8iF1oJ4L_fcgdQrp_YI&confirm=t&uuid=f28af9fb-9a6a-4d0b-be92-2cd5e0d9bec0
To: /content/spai/weights/spai.pth
100% 935M/935M [00:22<00:00, 42.4MB/s]
Weights: 934 MB  (expect ~934)


## 3. Download data

COCO val2017 for real images (~778 MB) + Synthbuster SD1.4 fakes (~12 GB).
Synthbuster download takes ~15 min — the cell checks if already present.

In [5]:
import os, glob

DATA = '/content/data'
os.makedirs(DATA, exist_ok=True)

# ── Try mounting Google Drive (for Synthbuster persistence) ────────────────
DRIVE_OK = False
try:
    from google.colab import drive
    drive.mount('/content/drive', timeout_ms=30000)
    DRIVE_DATA = '/content/drive/MyDrive/freqgen_data'
    os.makedirs(DRIVE_DATA, exist_ok=True)
    DRIVE_OK = True
    print('Google Drive mounted OK')
except BaseException as e:
    print(f'Drive mount failed ({e}) — Synthbuster will download to /content/ (not persistent)')
    DRIVE_DATA = DATA

# ── COCO val2017 (real images, 778 MB) ─────────────────────────────────────
COCO = f'{DATA}/coco_val2017'
if not os.path.exists(COCO) or len(os.listdir(COCO)) < 100:
    print('Downloading COCO val2017 (~778 MB)...')
    !wget -q -c 'http://images.cocodataset.org/zips/val2017.zip' -O /tmp/cv.zip
    !unzip -q /tmp/cv.zip -d {DATA}
    os.rename(f'{DATA}/val2017', COCO)
    os.remove('/tmp/cv.zip')
print(f'Real: {len(os.listdir(COCO))} COCO images')

# ── Synthbuster SD1.4 fakes (12 GB) ────────────────────────────────────────
DRIVE_SYNTH = f'{DRIVE_DATA}/synthbuster_sd14'
SYNTH = f'{DATA}/synthbuster/stable-diffusion-1-4'

if os.path.exists(DRIVE_SYNTH) and len(os.listdir(DRIVE_SYNTH)) >= 100:
    print(f'Synthbuster already on Drive: {len(os.listdir(DRIVE_SYNTH))} images')
    os.makedirs(os.path.dirname(SYNTH), exist_ok=True)
    if not os.path.exists(SYNTH):
        os.symlink(DRIVE_SYNTH, SYNTH)
elif os.path.exists(SYNTH) and len(os.listdir(SYNTH)) >= 100:
    print(f'Synthbuster already in /content/: {len(os.listdir(SYNTH))} images')
else:
    print('Downloading Synthbuster SD1.4 (~12 GB, ~15 min)...')
    !wget -L -c 'https://zenodo.org/records/10066460/files/synthbuster.zip' -O /tmp/sb.zip
    sz_mb = os.path.getsize('/tmp/sb.zip') // 1_000_000
    print(f'Downloaded: {sz_mb} MB')
    if sz_mb < 100:
        raise RuntimeError(f'Download failed ({sz_mb} MB).')
    # Extract SD1.4 only
    if DRIVE_OK:
        os.makedirs(DRIVE_SYNTH, exist_ok=True)
        !unzip -q /tmp/sb.zip 'synthbuster/stable-diffusion-1-4/*' -d /tmp/sb_extract
        import shutil
        for f in os.listdir('/tmp/sb_extract/synthbuster/stable-diffusion-1-4/'):
            shutil.move(f'/tmp/sb_extract/synthbuster/stable-diffusion-1-4/{f}', f'{DRIVE_SYNTH}/{f}')
        shutil.rmtree('/tmp/sb_extract', ignore_errors=True)
        os.makedirs(os.path.dirname(SYNTH), exist_ok=True)
        if not os.path.exists(SYNTH):
            os.symlink(DRIVE_SYNTH, SYNTH)
        print(f'Synthbuster saved to Drive: {len(os.listdir(DRIVE_SYNTH))} images')
    else:
        os.makedirs(os.path.dirname(SYNTH), exist_ok=True)
        !unzip -q /tmp/sb.zip 'synthbuster/stable-diffusion-1-4/*' -d {DATA}
        os.remove('/tmp/sb.zip')
        print(f'Synthbuster extracted to /content/: {len(os.listdir(SYNTH))} images')

FAKE_DIR = SYNTH
fake_n = len(glob.glob(f'{FAKE_DIR}/*.png')) if os.path.exists(FAKE_DIR) else 0
print(f'Fake: {fake_n} Synthbuster SD1.4 images ready')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted OK
Real: 5000 COCO images
--2026-06-23 10:21:46--  https://zenodo.org/records/10066460/files/synthbuster.zip
Resolving zenodo.org (zenodo.org)... 137.138.153.219, 188.184.98.114, 188.185.43.153, ...
Connecting to zenodo.org (zenodo.org)|137.138.153.219|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12372557226 (12G) [application/octet-stream]
Saving to: ‘/tmp/sb.zip’

/tmp/sb.zip         100%[===================>]  11.52G  22.2MB/s    in 53m 47s 

2026-06-23 11:15:34 (3.66 MB/s) - ‘/tmp/sb.zip’ saved [12372557226/12372557226]

Downloaded: 12372 MB
Synthbuster saved to Drive: 1000 images
Fake: 1000 Synthbuster SD1.4 images ready


## 4. Spectral matching attack

Run the freqgen attack: rewrite each fake's Fourier magnitude to match the
real spectral target. Phase (structure/content) is preserved.

In [ ]:
import numpy as np, glob, os
from PIL import Image
from tqdm.notebook import tqdm

SIZE = 256
DATA = '/content/data'
COCO = f'{DATA}/coco_val2017'
# Use real Synthbuster if available, otherwise pseudo_fakes
_synth = f'{DATA}/synthbuster/stable-diffusion-1-4'
import glob as _g
FAKE_DIR = _synth if (os.path.exists(_synth) and len(_g.glob(f'{_synth}/*.png')) >= 30) else f'{DATA}/pseudo_fakes'
print(f'FAKE_DIR: {FAKE_DIR}  ({len(_g.glob(f"{FAKE_DIR}/*.png"))} images)')

def load_gray(p):
    return np.array(Image.open(p).convert('L').resize((SIZE, SIZE)), dtype=np.float64)

def radial_profile(ch):
    F = np.fft.fftshift(np.fft.fft2(ch)); mag = np.abs(F)
    cy, cx = ch.shape[0]//2, ch.shape[1]//2
    y, x = np.ogrid[:ch.shape[0], :ch.shape[1]]
    r = np.round(np.sqrt((y-cy)**2 + (x-cx)**2)).astype(int)
    mr = min(ch.shape)//2
    t = np.bincount(r.ravel(), weights=mag.ravel())
    c = np.bincount(r.ravel())
    return t[:mr] / np.maximum(c[:mr], 1)

def spectral_match(img, target, gain_clip=(0.1, 12.0)):
    F = np.fft.fftshift(np.fft.fft2(img))
    cy, cx = img.shape[0]//2, img.shape[1]//2
    y, x = np.ogrid[:img.shape[0], :img.shape[1]]
    r = np.round(np.sqrt((y-cy)**2 + (x-cx)**2)).astype(int)
    mr = len(target)
    gain = np.clip(target / (radial_profile(img) + 1e-12), *gain_clip)
    gain[0] = 1.0
    gmap = gain[np.clip(r, 0, mr-1)]; gmap[r >= mr] = 1.0; gmap[r == 0] = 1.0
    return np.clip(np.fft.ifft2(np.fft.ifftshift(F * gmap)).real, 0, 255)

N = 30  # keep within T4 free-tier RAM
real_paths  = sorted(glob.glob(f'{COCO}/*.jpg'))[:N]
fake_paths  = sorted(glob.glob(f'{FAKE_DIR}/*.png'))[:N]
print(f'real: {len(real_paths)}  fake: {len(fake_paths)}')

print('Building real spectral target...')
target = np.mean([radial_profile(load_gray(p)) for p in tqdm(real_paths)], axis=0)

MATCHED = f'{DATA}/matched_sd14'
os.makedirs(MATCHED, exist_ok=True)
matched_paths = []
print('Running spectral matching attack...')
for p in tqdm(fake_paths):
    m = spectral_match(load_gray(p), target)
    out = f'{MATCHED}/{os.path.basename(p)}'
    Image.fromarray(m.astype(np.uint8)).save(out)
    matched_paths.append(out)
print(f'Saved {len(matched_paths)} matched fakes')

rh = np.mean([radial_profile(load_gray(p))[60:].mean() for p in real_paths])
fh = np.mean([radial_profile(load_gray(p))[60:].mean() for p in fake_paths])
mh = np.mean([radial_profile(load_gray(p))[60:].mean() for p in matched_paths])
print(f'\n=== SPECTRAL GAP ===')
print(f'High-band  real={rh:.1f}  fake={fh:.1f}  matched={mh:.1f}')
print(f'Gap real/fake={rh/fh:.2f}x    real/matched={rh/mh:.2f}x')


## 5. Prepare image directories for SPAI

SPAI inference takes a **directory** of images (from README). Copy/symlink
real, fake, and matched images into separate input folders.

In [ ]:
import os, shutil, glob
DATA = '/content/data'
COCO = f'{DATA}/coco_val2017'
FAKE_DIR = f'{DATA}/pseudo_fakes'
MATCHED = f'{DATA}/matched_sd14'

SPAI_IN = '/content/spai_input'
N = 30
for tag, paths in [
    ('real',    sorted(glob.glob(f'{COCO}/*.jpg'))[:N]),
    ('fake',    sorted(glob.glob(f'{FAKE_DIR}/*.png'))[:N]),
    ('matched', sorted(glob.glob(f'{MATCHED}/*.png'))[:N]),
]:
    d = f'{SPAI_IN}/{tag}'
    os.makedirs(d, exist_ok=True)
    for p in paths:
        dst = f'{d}/{os.path.basename(p)}'
        if not os.path.exists(dst):
            shutil.copy2(p, dst)
    print(f'{tag}: {len(os.listdir(d))} images')


## 6. SPAI inference — does the attack fool the SOTA detector?

Using the exact command from the README. Working dir must be `/content/spai`
so that `./weights/spai.pth` and `./configs/spai.yaml` resolve correctly.

In [ ]:
import os
os.makedirs('/content/spai_output/real',    exist_ok=True)
os.makedirs('/content/spai_output/fake',    exist_ok=True)
os.makedirs('/content/spai_output/matched', exist_ok=True)

%cd /content/spai

print('Running SPAI on REAL images...')
!python -m spai infer \
    --input /content/spai_input/real \
    --output /content/spai_output/real

print('Running SPAI on FAKE images...')
!python -m spai infer \
    --input /content/spai_input/fake \
    --output /content/spai_output/fake

print('Running SPAI on MATCHED (attacked) fakes...')
!python -m spai infer \
    --input /content/spai_input/matched \
    --output /content/spai_output/matched

print('SPAI inference done')

## 7. Results — evasion table

In [ ]:
import pandas as pd, glob, numpy as np

def load_scores(tag):
    csvs = glob.glob(f'/content/spai_output/{tag}/**/*.csv', recursive=True)
    if not csvs:
        raise FileNotFoundError(f'No SPAI output for {tag}.')
    df = pd.read_csv(csvs[0])
    print(f'{tag}: {len(df)} rows, columns: {list(df.columns)}')
    return df

res_real    = load_scores('real')
res_fake    = load_scores('fake')
res_matched = load_scores('matched')

# SPAI outputs a column named 'spai' (confirmed from actual run)
score_col = 'spai'
print(f'Score range fake: {res_fake[score_col].min():.3f} - {res_fake[score_col].max():.3f}')

fake_det    = (res_fake[score_col]    >= 0.5).mean()
matched_det = (res_matched[score_col] >= 0.5).mean()
real_fp     = (res_real[score_col]    >= 0.5).mean()

print()
print('='*55)
print('freqgen - SPAI Evasion Table (v2)')
print('='*55)
print(f'Spectral gap  real/fake={rh/fh:.1f}x  real/matched={rh/mh:.2f}x')
print(f'SPAI detects raw fakes:       {fake_det:.0%}')
print(f'SPAI detects matched fakes:   {matched_det:.0%}')
print(f'SPAI false-positives real:    {real_fp:.0%}')
print(f'Evasion rate (attack):        {1-matched_det:.0%}')
print('='*55)
if 1-matched_det > 0.5:
    print('RESULT: Attack EVADES SPAI -> gap found in CVPR 2025 SOTA')
elif 1-matched_det > 0.2:
    print('RESULT: Partial evasion')
else:
    print('RESULT: SPAI survives attack -> learned detectors are robust')
print()
print('Score distributions:')
print(f'  Real    mean={res_real[score_col].mean():.3f}  std={res_real[score_col].std():.3f}')
print(f'  Fake    mean={res_fake[score_col].mean():.3f}  std={res_fake[score_col].std():.3f}')
print(f'  Matched mean={res_matched[score_col].mean():.3f}  std={res_matched[score_col].std():.3f}')
